In [7]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import product

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import FastICA
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score






In [2]:
SEED = 25
np.random.seed(SEED)


In [3]:
X_train_df = pd.read_csv('./X_train.csv', skiprows=1, header=None)
y_train_df = pd.read_csv('./y_train.csv', skiprows=1, header=None)
X_test_df = pd.read_csv('./X_test.csv', skiprows=1, header=None)

X_train = X_train_df.values[:, 1:]
y_train = y_train_df.values[:, 1:]
X_test = X_test_df.values[:, 1:]

print(X_train.shape, y_train.shape, X_test.shape)


(1212, 832) (1212, 1) (776, 832)


In [4]:
def remove_outliers(
    X_train,
    y_train,
    X_test=None,
    contamination=0.8,     
):

    X_train = pd.DataFrame(X_train).copy()
    y_train = pd.Series(np.asarray(y_train).reshape(-1), index=X_train.index)

    if X_test is not None:
        X_test = pd.DataFrame(X_test).copy()

    n = len(X_train)


    #median imputation
    med = X_train.median(axis=0)
    Xtr_imp = X_train.fillna(med)
    Xte_imp = X_test.fillna(med) if X_test is not None else None

    miss_before = int(X_train.isna().sum().sum())
    print(f"[IMPUTE] Median imputation. Missing before: {miss_before}")

    #scaling
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xtr_std = scaler.fit_transform(Xtr_imp)

    #2D latent embedding (PLS)
    Z = None
    pls = PLSRegression(n_components=2)
    Z, _ = pls.fit_transform(Xtr_std, y_train.values.reshape(-1, 1))

    print("[EMBED] PLSRegression to 2D")
 

    #Isolation Forest on 2D space
    iso = IsolationForest(
        contamination=contamination,
        random_state=SEED,
        n_jobs=-1
    )
    pred = iso.fit_predict(Z)  # 1=inlier, -1=outlier

    flag_outlier = (pred == -1)

    mask_inliers = pd.Series(~flag_outlier, index=X_train.index)
    n_out = int(flag_outlier.sum())

    print(f"[IF] Removed {n_out} outliers out of {n} "
            f"({100*n_out/n:.2f}%). target≈{100*contamination:.2f}%")

    X_train_inliers = X_train.loc[mask_inliers].copy()
    y_train_inliers = y_train.loc[mask_inliers].copy()

    return X_train_inliers, y_train_inliers, Xte_imp


In [5]:
def feature_engineering(
    X_train, y_train, X_test,
    top_k_corr=200,                  #numbers of features kept by correlation with target
    rf_keep=200,                     # numbers of features kept by RF importance
    rf_n_estimators=1000,
    rf_max_depth=None,
    rf_min_samples_leaf=1,
    rf_max_features="sqrt",      
    corr_feature_max=0.99,        
    low_var_threshold=0.1,

):


    X_train = pd.DataFrame(X_train).copy()
    X_test  = pd.DataFrame(X_test).copy()
    y_train = pd.Series(np.asarray(y_train).reshape(-1), index=X_train.index)


    #median imputation
    med = X_train.median(axis=0)
    X_train_imp = X_train.fillna(med)
    X_test_imp  = X_test.fillna(med)

    miss_before = int(X_train.isna().sum().sum())
    print(f"[IMPUTE] Median imputation. Missing before: {miss_before} -> after: 0")

    # standardize 
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_train_std = pd.DataFrame(
        scaler.fit_transform(X_train_imp), columns=X_train.columns, index=X_train.index
    )
    X_test_std = pd.DataFrame(
        scaler.transform(X_test_imp), columns=X_test.columns, index=X_test.index
    )

    print("[SCALE] StandardScaler applied (fit on train).")

    #Remove low-variance features
    vt = VarianceThreshold(threshold=low_var_threshold)
    X_train_vt = vt.fit_transform(X_train_std)
    kept_idx_vt = vt.get_support(indices=True)
    vt_cols = X_train_std.columns[kept_idx_vt]
    X_train_vt = pd.DataFrame(X_train_vt, columns=vt_cols, index=X_train_std.index)
    X_test_vt  = pd.DataFrame(vt.transform(X_test_std), columns=vt_cols, index=X_test_std.index)

    dropped_zero_var = [c for c in X_train_std.columns if c not in vt_cols]

    print(f"[FILTER] low-variance features: {len(dropped_zero_var)}")

    #Top-K by absolute Pearson correlation with target (on VT-filtered set)
    corrs = {}
    yv = y_train
    for c in vt_cols:
        x = X_train_vt[c]
        if x.std() == 0:
            corrs[c] = 0.0
            continue
        try:
            corrs[c] = float(np.corrcoef(x, yv)[0, 1])
        except Exception:
            corrs[c] = 0.0
    corr_s = pd.Series(corrs).abs().sort_values(ascending=False)
    corr_keep_cols = corr_s.head(min(top_k_corr, len(corr_s))).index.tolist()

    X_train_corr = X_train_vt[corr_keep_cols].copy()
    X_test_corr  = X_test_vt[corr_keep_cols].copy()

    print(f"[SELECT] Kept top-{len(corr_keep_cols)} features by |Pearson r| with target (requested {top_k_corr}).")

    #remove highly inter-correlated features (|r| >= corr_feature_max)
    if len(corr_keep_cols) > 1:
        cm = X_train_corr.corr().abs()
        ordered = list(corr_s.loc[corr_keep_cols].index)  # already sorted desc by |corr to y|
        kept, dropped = [], []
        for c in ordered:
            if not kept:
                kept.append(c)
            else:
                # check correlation with already kept features
                too_corr = any(cm.loc[c, k] >= corr_feature_max for k in kept if c != k)
                if too_corr:
                    dropped.append(c)
                else:
                    kept.append(c)

        X_train_decorr = X_train_corr[kept].copy()
        X_test_decorr  = X_test_corr[kept].copy()


        print(f"[DE-CORR] Removed {len(dropped)} highly inter-correlated features "
                f"(threshold |r| ≥ {corr_feature_max}).")

    else:
        X_train_decorr = X_train_corr
        X_test_decorr  = X_test_corr
        dropped = []

    # RandomForest-based selection, keep top 'rf_keep' features
    rf = RandomForestRegressor(
        n_estimators=rf_n_estimators,
        max_depth=rf_max_depth,
        min_samples_leaf=rf_min_samples_leaf,
        max_features=rf_max_features, 
        random_state=SEED,
        n_jobs=-1
    )
    rf.fit(X_train_decorr, y_train)
    importances = pd.Series(rf.feature_importances_, index=X_train_decorr.columns).sort_values(ascending=False)

    if rf_keep is None or rf_keep <= 0:
        final_cols = importances.index.tolist()
    else:
        final_cols = importances.head(min(rf_keep, len(importances))).index.tolist()

    X_train_final = X_train_decorr[final_cols].copy()
    X_test_final  = X_test_decorr[final_cols].copy()


    print(f"[RF] RandomForest feature selection:")
    print(f"     - kept {len(final_cols)} features (requested {rf_keep})")
    print(f"     - top-10 importances:\n{importances.head(10)}")
    
    return X_train_final, X_test_final


In [6]:
shuffled_indices = np.random.permutation(len(X_train))
X_train = X_train[shuffled_indices]
y_train = y_train[shuffled_indices]

In [8]:
grid = {
    "contamination": [0.045],
    "top_k_corr":    [200],
    "rf_keep":       [ 167],
    "c":             [  52],
}


kf = KFold(n_splits=10, shuffle=True, random_state=SEED)

results = [] 

for contamination, top_k_corr, rf_keep, c in product(
        grid["contamination"], grid["top_k_corr"], grid["rf_keep"],grid["c"]):
    try:

        X_train_in_i, y_train_in_i, X_test_tmp = remove_outliers(
            X_train=X_train, y_train=y_train, X_test=X_test,
            contamination=contamination,
        )


        X_train_fe_i, X_test_fe_i = feature_engineering(
            X_train=X_train_in_i, y_train=y_train_in_i, X_test=X_test_tmp,
            top_k_corr=top_k_corr,
            rf_keep=rf_keep,
        )


        fold_scores = []
        for tr_idx, te_idx in kf.split(X_train_fe_i):
            X_tr, X_te = X_train_fe_i.iloc[tr_idx], X_train_fe_i.iloc[te_idx]
            y_tr, y_te = y_train_in_i.iloc[tr_idx], y_train_in_i.iloc[te_idx]

            svr = SVR(kernel='rbf', C=c, gamma="scale")
            svr.fit(X_tr, y_tr)
            y_hat = svr.predict(X_te)
            fold_scores.append(r2_score(y_te, y_hat))

        mean_r2 = float(np.mean(fold_scores))
        std_r2  = float(np.std(fold_scores, ddof=1))
        results.append((mean_r2, std_r2, 
                        {"contamination": contamination,
                         "top_k_corr": top_k_corr,
                         "rf_keep": rf_keep,
                         "c": c},
                        fold_scores))
        print(f"[TRY] cont={contamination} | top_k={top_k_corr} | rf_keep={rf_keep}, c={c} "
              f"-> mean R2={mean_r2:.4f} ± {std_r2:.4f}")

    except Exception as e:
        print(f"[SKIP] cont={contamination}, top_k={top_k_corr}, rf_keep={rf_keep}, c={c} "
              f"-> error: {e}")
        continue


if not results:
    raise RuntimeError("No valid result.")


best = max(results, key=lambda t: t[0])
best_mean, best_std, best_params, best_scores = best

print("\n" + "="*60)
print("BEST PARAMS (by mean CV R²):")
print(f"  contamination: {best_params['contamination']}")
print(f"  top_k_corr   : {best_params['top_k_corr']}")
print(f"  rf_keep      : {best_params['rf_keep']}")
print(f"  c            : {best_params['c']}")
print(f"  mean R²      : {best_mean:.4f} ± {best_std:.4f}")
print(f"  fold R²s     : {np.round(best_scores, 4).tolist()}")
print("="*60)


[IMPUTE] Median imputation. Missing before: 76902
[EMBED] PLSRegression to 2D
[IF] Removed 55 outliers out of 1212 (4.54%). target≈4.50%
[IMPUTE] Median imputation. Missing before: 73449 -> after: 0
[SCALE] StandardScaler applied (fit on train).
[FILTER] low-variance features: 4
[SELECT] Kept top-200 features by |Pearson r| with target (requested 200).
[DE-CORR] Removed 0 highly inter-correlated features (threshold |r| ≥ 0.99).
[RF] RandomForest feature selection:
     - kept 167 features (requested 167)
     - top-10 importances:
115    0.052598
415    0.037422
159    0.034455
458    0.031680
507    0.031334
485    0.025951
133    0.024943
702    0.023192
465    0.023186
641    0.023170
dtype: float64
[TRY] cont=0.045 | top_k=200 | rf_keep=167, c=52 -> mean R2=0.6849 ± 0.0328

BEST PARAMS (by mean CV R²):
  contamination: 0.045
  top_k_corr   : 200
  rf_keep      : 167
  c            : 52
  mean R²      : 0.6849 ± 0.0328
  fold R²s     : [0.6914, 0.7237, 0.6977, 0.6979, 0.6952, 0.7359

best 0.6950

In [605]:
best_params

{'contamination': 0.045, 'top_k_corr': 200, 'rf_keep': 167, 'c': 52}

In [606]:
X_train_in_i, y_train_in_i, X_test_tmp = remove_outliers(
    X_train=X_train, y_train=y_train, X_test=X_test,
    contamination=best_params['contamination'],
)


X_train_fe_i, X_test_fe_i = feature_engineering(
    X_train=X_train_in_i, y_train=y_train_in_i, X_test=X_test_tmp,
    top_k_corr=best_params['top_k_corr'],
    rf_keep=best_params['rf_keep'],
)

svr = SVR(kernel='rbf', C=best_params['c'], gamma="scale")
svr.fit(X_train_fe_i, y_train_in_i)
y_test_pred = svr.predict(X_test_fe_i)  
table = pd.DataFrame({'id': np.arange(0, y_test_pred.shape[0]), 'y': y_test_pred.flatten()})
table.to_csv('submission_grid_search.csv', index=False)


[IMPUTE] Median imputation. Missing before: 76902
[EMBED] PLSRegression to 2D
[IF] Removed 55 outliers out of 1212 (4.54%). target≈4.50%
[IMPUTE] Median imputation. Missing before: 73434 -> after: 0
[SCALE] StandardScaler applied (fit on train).
[FILTER] low-variance features: 4
[SELECT] Kept top-200 features by |Pearson r| with target (requested 200).
[DE-CORR] Removed 0 highly inter-correlated features (threshold |r| ≥ 0.99).
[RF] RandomForest feature selection:
     - kept 167 features (requested 167)
     - top-10 importances:
115    0.049723
415    0.041441
458    0.031159
507    0.028317
159    0.028133
702    0.025565
641    0.024999
133    0.024664
334    0.023897
194    0.023200
dtype: float64


In [18]:
X_train_in, y_train_in, X_test_tmp = remove_outliers(
    X_train=X_train, y_train=y_train, X_test=X_test,
    contamination=best_params['contamination'],
)


X_train_fe, X_test_fe = feature_engineering(
    X_train=X_train_in, y_train=y_train_in, X_test=X_test_tmp,
    top_k_corr=best_params['top_k_corr'],
    rf_keep=best_params['rf_keep'],
)

[IMPUTE] Median imputation. Missing before: 76902
[EMBED] PLSRegression to 2D
[IF] Removed 55 outliers out of 1212 (4.54%). target≈4.50%
[IMPUTE] Median imputation. Missing before: 73449 -> after: 0
[SCALE] StandardScaler applied (fit on train).
[FILTER] low-variance features: 4
[SELECT] Kept top-200 features by |Pearson r| with target (requested 200).
[DE-CORR] Removed 0 highly inter-correlated features (threshold |r| ≥ 0.99).
[RF] RandomForest feature selection:
     - kept 167 features (requested 167)
     - top-10 importances:
115    0.052598
415    0.037422
159    0.034455
458    0.031680
507    0.031334
485    0.025951
133    0.024943
702    0.023192
465    0.023186
641    0.023170
dtype: float64


In [21]:
X_train_local, X_test_local, y_train_local, y_test_local = train_test_split(X_train_fe, y_train_in, test_size=0.2, random_state=SEED)

In [22]:
svr = SVR(kernel='rbf', C=200, gamma="scale")
svr.fit(X_train_local, y_train_local)
print("R² test (SVR):", r2_score(y_test_local, svr.predict(X_test_local)))

R² test (SVR): 0.6956462392007849


In [23]:
y_test_pred = svr.predict(X_test_fe)

table = pd.DataFrame({'id': np.arange(0, y_test_pred.shape[0]), 'y': y_test_pred.flatten()})
table.to_csv('submission.csv', index=False)

best 0.7373812788914935

FOLLOWING MODLES (LESSS GOOD)

In [612]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

input_dim = X_train_local.shape[1]


def build_model(input_dim, width=80, depth=10, dropout_rate=0.2, lr=0.01, alpha=0.01):
    model = keras.Sequential()
  
    model.add(layers.Dense(width, input_shape=(input_dim,)))
    model.add(layers.LeakyReLU(alpha=alpha))
    model.add(layers.Dropout(dropout_rate))

    for _ in range(depth - 1):
        model.add(layers.Dense(width))
        model.add(layers.LeakyReLU(alpha=alpha))
        model.add(layers.Dropout(dropout_rate))
 
    model.add(layers.Dense(1))
    opt = keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss="mse", metrics=["mse"])
    return model

nn = build_model(
    input_dim=input_dim,
    width=80,
    depth=10,
    dropout_rate=0.1,   
    lr=0.001,
    alpha=0.01        
)

history = nn.fit(
    X_train_local, y_train_local,
    epochs=100,
    batch_size=32,       
    shuffle=True,        
    validation_data=(X_test_local, y_test_local),
    verbose=1
)

print("Training complete.")


/home/celestin/g/eth/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/celestin/g/eth/.venv/lib/python3.12/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - loss: 3142.8071 - mse: 3142.8071 - val_loss: 1062.4061 - val_mse: 1062.4061
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1033.2893 - mse: 1033.2893 - val_loss: 606.9175 - val_mse: 606.9175
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 574.8203 - mse: 574.8203 - val_loss: 393.8341 - val_mse: 393.8341
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 380.7934 - mse: 380.7934 - val_loss: 316.3308 - val_mse: 316.3308
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 322.7706 - mse: 322.7706 - val_loss: 241.7673 - val_mse: 241.7673
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 264.2929 - mse: 264.2929 - val_loss: 298.5059 - val_mse: 298.5059
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 259.8705 - mse: 259.8705 - val_loss: 259.7590 - val_mse: 259.7590
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 227.6654 - mse: 227.6654 - val_loss: 312.0952 - val_mse:

In [613]:



y_pred_train_nn = nn.predict(X_train_local).ravel()
y_pred_test_nn  = nn.predict(X_test_local).ravel()

r2_train_nn = r2_score(y_train_local, y_pred_train_nn)
r2_test_nn  = r2_score(y_test_local,  y_pred_test_nn)

print("R² scores (Neural Network):")
print(f"  • Train local: {r2_train_nn:.4f}")
print(f"  • Test  local: {r2_test_nn:.4f}")

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
R² scores (Neural Network):
  • Train local: -0.7286
  • Test  local: -0.9245


In [614]:

from sklearn.ensemble import ExtraTreesRegressor
etr = ExtraTreesRegressor(n_estimators=1000, max_depth=None, random_state=42, n_jobs=-1)
etr.fit(X_train_local, y_train_local)
print("R² test (ExtraTrees):", r2_score(y_test_local, etr.predict(X_test_local)))


R² test (ExtraTrees): 0.595919983801865


In [615]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=1000, random_state=42, n_jobs=-1)
rf.fit(X_train_local, y_train_local)
print("R² test (RandomForest):", r2_score(y_test_local, rf.predict(X_test_local)))

R² test (RandomForest): 0.5872766211035376


In [616]:
from sklearn.ensemble import GradientBoostingRegressor
gbr = GradientBoostingRegressor(n_estimators=1000, learning_rate=0.01, max_depth=5, random_state=42)
gbr.fit(X_train_local, y_train_local)
print("R² test (GradientBoosting):", r2_score(y_test_local, gbr.predict(X_test_local)))

R² test (GradientBoosting): 0.6266809548912706


In [617]:
# XGBoost
from xgboost import XGBRegressor
xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train_local, y_train_local)
print("R² test (XGBoost):", r2_score(y_test_local, xgb.predict(X_test_local)))

R² test (XGBoost): 0.6218000001530265


In [618]:
# LightGBM
from lightgbm import LGBMRegressor
lgb = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.02,
    num_leaves=64,
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
lgb.fit(X_train_local, y_train_local)
print("R² test (LightGBM):", r2_score(y_test_local, lgb.predict(X_test_local)))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023798 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 50934
[LightGBM] [Info] Number of data points in the train set: 765, number of used features: 200
[LightGBM] [Info] Start training from score 69.809150
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

In [ ]:
# CatBoost 
from catboost import CatBoostRegressor
cat = CatBoostRegressor(
    iterations=1000,
    depth=6,
    learning_rate=0.01,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)
cat.fit(X_train_local, y_train_local)
print("R² test (CatBoost):", r2_score(y_test_local, cat.predict(X_test_local)))

R² test (CatBoost): 0.594514918402645


In [620]:
from sklearn.linear_model import HuberRegressor
huber = HuberRegressor(epsilon=1.35, alpha=0.0001)
huber.fit(X_train_local, y_train_local)
print("R² test (HuberRegressor):", r2_score(y_test_local, huber.predict(X_test_local)))

R² test (HuberRegressor): 0.322239015824908


/home/celestin/g/eth/.venv/lib/python3.12/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [621]:
from sklearn.svm import SVR
svr = SVR(kernel='rbf', C=30, gamma='scale')
svr.fit(X_train_local, y_train_local)
print("R² test (SVR):", r2_score(y_test_local, svr.predict(X_test_local)))

R² test (SVR): 0.6500660251304864


In [622]:
from sklearn.neighbors import KNeighborsRegressor
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_local, y_train_local)
print("R² test (KNN):", r2_score(y_test_local, knn.predict(X_test_local)))

R² test (KNN): 0.5205038972267166


In [623]:
from tensorflow.keras import layers, models, regularizers

model = models.Sequential([
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.3),
    layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.fit(X_train_local, y_train_local, validation_data=(X_test_local, y_test_local),
          epochs=100, batch_size=32, verbose=1)
y_pred_train_nn = model.predict(X_train_local).ravel()
y_pred_test_nn  = model.predict(X_test_local).ravel()
print("R² test (Neural Network):", r2_score(y_test_local, y_pred_test_nn))

Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 4251.9707 - val_loss: 3180.3884
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 2104.1580 - val_loss: 1168.6127
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1198.7056 - val_loss: 907.2556
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 869.3432 - val_loss: 700.0925
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 714.8572 - val_loss: 600.4825
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 642.4526 - val_loss: 534.7379
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 524.5079 - val_loss: 479.7301
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 461.7668 - val_loss: 411.6763
Epoch 9/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 406.8882 - val_loss: 376.8336
Epoch 10/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 348.2354 - val_loss: 319.3230
Epoch 11/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 303.6188 - val_loss: 288.0735
Epoch 12/100
